# Project Risk Prediction — Full ML Training Notebook

Complete machine learning workflow for the PFE project: EDA → Preprocessing → Feature Engineering → Baseline Comparison → Hyperparameter Tuning → Ensemble → Cross-Validation → Model Selection & Save.

| Step | Description |
|------|-------------|
| 1 | **Data Exploration** — Load and inspect the 4,000-row real dataset |
| 2 | **Preprocessing** — Encode categoricals, merge Critical→High |
| 3 | **Feature Engineering** — Add 3 derived features (13 total) |
| 4 | **Train/Test Split** — Stratified 80/20 + StandardScaler |
| 5 | **Baseline Comparison** — 9 algorithms on 10 vs 13 features |
| 6 | **Hyperparameter Tuning** — RandomizedSearchCV (n_iter=25, cv=5) |
| 7 | **Ensemble Methods** — Soft Voting + Stacking Classifier |
| 8 | **Cross-Validation** — 5-fold stratified robustness check |
| 9 | **Model Selection & Save** — Persist best model to `models/` |

---

**Dataset**: `project_risk_raw_dataset.csv` — 4,000 real project records, 51 columns  
**Target**: `Risk_Level` → Low / Medium / High (Critical merged → High)  
**Previous best**: RandomForest 54.50% on 10 features  
**Goal**: Improve accuracy via feature engineering, systematic tuning and ensembles

---
## 0. Setup — Install Optional Dependencies
Install LightGBM (leaf-wise gradient boosting, typically outperforms XGBoost on small tabular datasets).

In [ ]:
import subprocess, sys
r = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'lightgbm', '-q'],
    capture_output=True, text=True
)
print('LightGBM:', 'installed' if r.returncode == 0 else f'failed ({r.stderr[:120]})')

## Imports

In [ ]:
import os, json, copy, time, warnings, sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model   import LogisticRegression
from sklearn.tree           import DecisionTreeClassifier
from sklearn.neighbors      import KNeighborsClassifier
from sklearn.svm            import SVC
from sklearn.ensemble       import (
    RandomForestClassifier, ExtraTreesClassifier,
    GradientBoostingClassifier, VotingClassifier, StackingClassifier
)
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
)
from sklearn.preprocessing  import StandardScaler
from sklearn.base           import clone
from sklearn.metrics        import accuracy_score, classification_report, confusion_matrix
from scipy.stats            import randint, uniform
from xgboost                import XGBClassifier

try:
    from lightgbm import LGBMClassifier
    LGBM_AVAILABLE = True
    print('LightGBM : available')
except ImportError:
    LGBM_AVAILABLE = False
    print('LightGBM : NOT available — skipping LGBM models')

warnings.filterwarnings('ignore')
np.random.seed(42)
print(f'Python {sys.version.split()[0]}  |  numpy {np.__version__}  |  pandas {pd.__version__}')

## Configuration
> **Important**: Run this notebook with the Jupyter kernel started from the `ml/` directory,  
> OR change `ML_DIR` below to the absolute path of your `ml/` folder.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
ML_DIR     = Path(os.getcwd())            # should be the ml/ directory
CSV_PATH   = ML_DIR / 'project_risk_raw_dataset.csv'
MODELS_DIR = ML_DIR / 'models'
MODELS_DIR.mkdir(exist_ok=True)

if not CSV_PATH.exists():
    # Fallback: try to find the file relative to a known location
    ML_DIR = Path('D:/Hamza pfe/pfe/ml')
    CSV_PATH   = ML_DIR / 'project_risk_raw_dataset.csv'
    MODELS_DIR = ML_DIR / 'models'
    MODELS_DIR.mkdir(exist_ok=True)

assert CSV_PATH.exists(), f'CSV not found at {CSV_PATH}. Set ML_DIR manually.'
print(f'ML directory : {ML_DIR}')
print(f'Dataset      : {CSV_PATH.name}  (exists={CSV_PATH.exists()})')
print(f'Models dir   : {MODELS_DIR}')

# ── Column mappings ────────────────────────────────────────────────────────────
CSV_COL_MAP = {
    'Team_Size':                      'team_size',
    'Project_Budget_USD':             'budget_usd',
    'Estimated_Timeline_Months':      'duration_months',
    'Complexity_Score':               'complexity_score',
    'Stakeholder_Count':              'stakeholder_count',
    'Previous_Delivery_Success_Rate': 'success_rate',
    'Budget_Utilization_Rate':        'budget_utilization',
    'Team_Experience_Level':          'team_experience',
    'Requirement_Stability':          'requirement_stability',
    'External_Dependencies_Count':    'external_dependencies',
}
EXPERIENCE_MAP = {'Junior': 0, 'Mixed': 1, 'Senior': 2, 'Expert': 3}
STABILITY_MAP  = {'Volatile': 0, 'Moderate': 1, 'Stable': 2}
LABEL_MAP      = {'Low': 0, 'Medium': 1, 'High': 2, 'Critical': 2}
RISK_LABELS    = {0: 'Low', 1: 'Medium', 2: 'High'}

# ── Feature sets ───────────────────────────────────────────────────────────────
FEATURES_BASE = [
    'team_size', 'budget_usd', 'duration_months', 'complexity_score',
    'stakeholder_count', 'success_rate', 'budget_utilization',
    'team_experience', 'requirement_stability', 'external_dependencies'
]
# 3 engineered features (computed from base inputs — no extra form fields needed)
FEATURES_ENG = FEATURES_BASE + ['budget_per_person', 'timeline_pressure', 'team_effectiveness']

print(f'\nBase feature set : {len(FEATURES_BASE)} features')
print(f'Eng. feature set : {len(FEATURES_ENG)} features  (+3 derived)')

---
## Step 1 — Data Exploration
Load the raw CSV. Inspect shape, column types, null counts and class balance.

In [ ]:
df_raw = pd.read_csv(CSV_PATH)
print(f'Shape   : {df_raw.shape[0]} rows × {df_raw.shape[1]} columns')
print(f'Columns : {list(df_raw.columns[:8])} ...')

print('\nRisk_Level distribution:')
vc = df_raw['Risk_Level'].value_counts()
for label, count in vc.items():
    bar = '#' * (count // 30)
    print(f'  {label:10}  {count:5d}  {bar}')

df_raw[list(CSV_COL_MAP.keys()) + ['Risk_Level']].head(3)

In [ ]:
needed_cols = list(CSV_COL_MAP.keys()) + ['Risk_Level']

print('Null counts in selected columns:')
nulls = df_raw[needed_cols].isnull().sum()
print(nulls[nulls > 0].to_string() if nulls.any() else '  None — dataset is clean!')

print('\nNumeric statistics:')
num_cols = [c for c in needed_cols if df_raw[c].dtype in ['float64', 'int64']]
df_raw[num_cols].describe().round(2)

---
## Step 2 — Preprocessing

| Action | Detail |
|--------|--------|
| Select columns | 10 feature columns + Risk_Level |
| Encode experience | Junior→0, Mixed→1, Senior→2, Expert→3 (ordinal) |
| Encode stability | Volatile→0, Moderate→1, Stable→2 (ordinal) |
| Merge labels | Critical → High (3-class model: Low/Medium/High) |
| Drop nulls | Remove any incomplete rows |

In [ ]:
df = df_raw[list(CSV_COL_MAP.keys()) + ['Risk_Level']].copy()
df.rename(columns=CSV_COL_MAP, inplace=True)

df['team_experience']       = df['team_experience'].map(EXPERIENCE_MAP)
df['requirement_stability'] = df['requirement_stability'].map(STABILITY_MAP)
df['risk']                  = df['Risk_Level'].map(LABEL_MAP)

before = len(df)
df.dropna(inplace=True)
print(f'Rows before dropna : {before}')
print(f'Rows after  dropna : {len(df)}  (dropped {before - len(df)})')

print('\nFinal class distribution:')
dist = df['risk'].value_counts().sort_index()
for k, v in dist.items():
    pct = v / len(df) * 100
    print(f'  {RISK_LABELS[k]:8} : {v:5d}  ({pct:.1f}%)')

---
## Step 3 — Feature Engineering

Three interaction features derived from the 10 base inputs:

| New Feature | Formula | Risk Intuition |
|-------------|---------|----------------|
| `budget_per_person` | `budget_usd / team_size` | Low value → stretched resources → higher risk |
| `timeline_pressure` | `complexity_score / duration_months` | High value → complex work per time unit → higher risk |
| `team_effectiveness` | `team_experience × success_rate` | Low value → inexperienced + poor track record → higher risk |

These encode non-linear interaction effects that single features cannot represent individually.  
They are **computed automatically in `predict.py`** from the 10 user inputs — no extra form fields required.

In [ ]:
df['budget_per_person']  = df['budget_usd'] / df['team_size'].clip(lower=1)
df['timeline_pressure']  = df['complexity_score'] / df['duration_months'].clip(lower=1)
df['team_effectiveness'] = df['team_experience'] * df['success_rate']

print('Engineered feature statistics:')
df[['budget_per_person', 'timeline_pressure', 'team_effectiveness']].describe().round(3)

---
## Step 4 — Train / Test Split & Scaling

- **Stratified 80/20 split** preserves class proportions in both sets  
- `StandardScaler` is **fitted on training data only** → zero data leakage  
- Two parallel scaled datasets: base 10 features and engineered 13 features  
  (same row indices, so comparisons are fair)

In [ ]:
y       = df['risk'].values.astype(int)
indices = np.arange(len(y))

tr_idx, te_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_base = df[FEATURES_BASE].values
X_eng  = df[FEATURES_ENG].values

X_base_tr, X_base_te = X_base[tr_idx], X_base[te_idx]
X_eng_tr,  X_eng_te  = X_eng[tr_idx],  X_eng[te_idx]
y_tr, y_te           = y[tr_idx],      y[te_idx]

scaler_base = StandardScaler().fit(X_base_tr)
scaler_eng  = StandardScaler().fit(X_eng_tr)

Xs_base_tr = scaler_base.transform(X_base_tr)
Xs_base_te = scaler_base.transform(X_base_te)
Xs_eng_tr  = scaler_eng.transform(X_eng_tr)
Xs_eng_te  = scaler_eng.transform(X_eng_te)

print(f'Training set  : {len(y_tr)} samples')
print(f'Test set      : {len(y_te)} samples')
print(f'Base features : {X_base_tr.shape[1]}')
print(f'Eng. features : {X_eng_tr.shape[1]}')
print(f'\nTrain class counts : {Counter(y_tr)}')
print(f'Test  class counts : {Counter(y_te)}')

---
## Step 5 — Baseline Model Comparison

Train **9 classifiers** head-to-head on both feature sets with default parameters.
The `Delta` column shows how much the 3 engineered features help each algorithm.

| Algorithm | Type | Typical strength |
|-----------|------|------------------|
| LogisticReg | Linear | Fast baseline, good for linearly separable problems |
| DecisionTree | Single tree | Interpretable, prone to overfit |
| KNN | Distance-based | Good when clusters are tight |
| RandomForest | Bagging (current winner) | Robust, handles noise well |
| ExtraTrees | Randomised bagging | Faster than RF, similar accuracy |
| GradBoost | Sequential boosting | Accurate, slow training |
| XGBoost | Optimised boosting | Previous competitor |
| SVC-RBF | Margin-based | Strong on small datasets |
| LightGBM | Leaf-wise boosting | Usually fastest & most accurate on tabular data |

In [ ]:
def make_models():
    m = {
        'LogisticReg'  : LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42),
        'DecisionTree' : DecisionTreeClassifier(max_depth=8, class_weight='balanced', random_state=42),
        'KNN'          : KNeighborsClassifier(n_neighbors=7),
        'RandomForest' : RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                                random_state=42, n_jobs=-1),
        'ExtraTrees'   : ExtraTreesClassifier(n_estimators=300, class_weight='balanced',
                                               random_state=42, n_jobs=-1),
        'GradBoost'    : GradientBoostingClassifier(n_estimators=200, max_depth=4, random_state=42),
        'XGBoost'      : XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05,
                                       subsample=0.8, eval_metric='mlogloss',
                                       random_state=42, n_jobs=-1, verbosity=0),
        'SVC-RBF'      : SVC(kernel='rbf', C=1.0, class_weight='balanced',
                             probability=True, random_state=42),
    }
    if LGBM_AVAILABLE:
        m['LightGBM'] = LGBMClassifier(n_estimators=400, learning_rate=0.05,
                                       class_weight='balanced', random_state=42,
                                       n_jobs=-1, verbose=-1)
    return m

baseline_results = {}
print(f'{"Model":<16}  {"10 features":>11}  {"13 features":>11}  {"Delta":>8}')
print('-' * 56)

for name, model in make_models().items():
    m10 = clone(model)
    m10.fit(Xs_base_tr, y_tr)
    a10 = accuracy_score(y_te, m10.predict(Xs_base_te))

    m13 = clone(model)
    m13.fit(Xs_eng_tr, y_tr)
    a13 = accuracy_score(y_te, m13.predict(Xs_eng_te))

    baseline_results[name] = {'a10': a10, 'a13': a13, 'm10': m10, 'm13': m13}
    delta = (a13 - a10) * 100
    print(f'{name:<16}  {a10*100:>9.2f}%  {a13*100:>9.2f}%  {delta:>+7.2f}%')

print('-' * 56)
best_bl_name = max(baseline_results,
                   key=lambda n: max(baseline_results[n]['a10'], baseline_results[n]['a13']))
best_bl_acc  = max(baseline_results[best_bl_name]['a10'], baseline_results[best_bl_name]['a13'])
print(f'\nBest baseline model : {best_bl_name}  ({best_bl_acc*100:.2f}%)')

---
## Step 6 — Hyperparameter Tuning

Apply `RandomizedSearchCV` to the most promising models:

- **n_iter = 25** random parameter combinations per model
- **cv = 5** stratified k-fold cross-validation
- **scoring = accuracy**
- **n_jobs = -1** (parallel, all CPU cores)
- All tuning uses the **13-feature engineered dataset**

> ⏱ Expected runtime: 3–8 minutes depending on your machine.

In [ ]:
CV5 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

PARAM_SPACES = {
    'RandomForest': {
        'estimator': RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        'params': {
            'n_estimators'     : randint(200, 600),
            'max_depth'        : [None, 8, 12, 16, 20],
            'min_samples_split': randint(2, 10),
            'min_samples_leaf' : randint(1, 6),
            'max_features'     : ['sqrt', 'log2', 0.5, 0.7],
        }
    },
    'ExtraTrees': {
        'estimator': ExtraTreesClassifier(class_weight='balanced', random_state=42, n_jobs=-1),
        'params': {
            'n_estimators'     : randint(200, 600),
            'max_depth'        : [None, 8, 12, 16],
            'min_samples_split': randint(2, 10),
            'min_samples_leaf' : randint(1, 6),
            'max_features'     : ['sqrt', 'log2', 0.6],
        }
    },
    'GradBoost': {
        'estimator': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators'     : randint(200, 500),
            'max_depth'        : randint(3, 7),
            'learning_rate'    : uniform(0.03, 0.15),
            'subsample'        : uniform(0.6, 0.4),
            'min_samples_split': randint(2, 10),
            'max_features'     : ['sqrt', 'log2', 0.5],
        }
    },
    'XGBoost': {
        'estimator': XGBClassifier(eval_metric='mlogloss', random_state=42,
                                   n_jobs=-1, verbosity=0),
        'params': {
            'n_estimators'    : randint(300, 700),
            'max_depth'       : randint(3, 9),
            'learning_rate'   : uniform(0.01, 0.15),
            'subsample'       : uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.5, 0.5),
            'min_child_weight': randint(1, 6),
            'gamma'           : uniform(0, 0.3),
        }
    },
}

if LGBM_AVAILABLE:
    PARAM_SPACES['LightGBM'] = {
        'estimator': LGBMClassifier(class_weight='balanced', random_state=42,
                                    n_jobs=-1, verbose=-1),
        'params': {
            'n_estimators'   : randint(300, 700),
            'max_depth'      : randint(3, 9),
            'learning_rate'  : uniform(0.02, 0.13),
            'num_leaves'     : randint(20, 80),
            'subsample'      : uniform(0.6, 0.4),
            'colsample_bytree': uniform(0.5, 0.5),
            'reg_alpha'      : uniform(0, 1),
            'reg_lambda'     : uniform(0, 1),
        }
    }

print(f'Models to tune : {list(PARAM_SPACES.keys())}')
print(f'Feature set    : engineered 13 features')
print(f'CV strategy    : {CV5}')

In [ ]:
tuned_results = {}
total_start   = time.time()

for name, cfg in PARAM_SPACES.items():
    t0 = time.time()
    print(f'[{name}] Tuning...', end=' ', flush=True)

    search = RandomizedSearchCV(
        cfg['estimator'], cfg['params'],
        n_iter=25, cv=CV5, scoring='accuracy',
        random_state=42, n_jobs=-1, verbose=0
    )
    search.fit(Xs_eng_tr, y_tr)

    best = search.best_estimator_
    test_acc = accuracy_score(y_te, best.predict(Xs_eng_te))

    tuned_results[name] = {
        'model'      : best,
        'test_acc'   : test_acc,
        'cv_score'   : search.best_score_,
        'best_params': search.best_params_,
    }
    elapsed = time.time() - t0
    print(f'CV={search.best_score_*100:.2f}%  Test={test_acc*100:.2f}%  [{elapsed:.0f}s]')
    print(f'  Best params: {search.best_params_}\n')

print(f'Total tuning time: {time.time() - total_start:.0f}s')

---
## Step 7 — Ensemble Methods

### 7a. Soft Voting Classifier
Averages the class probabilities from the **top 3 tuned models**.  
Works best when models are diverse (different algorithm families).

### 7b. Stacking Classifier
Uses each tuned model as a base learner, then trains a **Logistic Regression meta-learner**  
on the out-of-fold predictions (5-fold internal CV).  
More powerful but also more expensive.

In [ ]:
# Pick top 3 tuned models
top3 = sorted(tuned_results.items(), key=lambda x: x[1]['test_acc'], reverse=True)[:3]
print('Top 3 tuned models selected for ensemble:')
for n, r in top3:
    print(f'  {n:<18}  test={r["test_acc"]*100:.2f}%  cv={r["cv_score"]*100:.2f}%')

estimators = [(n, r['model']) for n, r in top3]

# Soft Voting
voting = VotingClassifier(estimators=estimators, voting='soft', n_jobs=-1)
voting.fit(Xs_eng_tr, y_tr)
voting_acc = accuracy_score(y_te, voting.predict(Xs_eng_te))
print(f'\nSoft Voting Classifier  →  {voting_acc*100:.2f}%')

In [ ]:
# Stacking with LR meta-learner
stacking = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000, C=5.0, random_state=42),
    cv=5, n_jobs=-1, passthrough=False
)
stacking.fit(Xs_eng_tr, y_tr)
stacking_acc = accuracy_score(y_te, stacking.predict(Xs_eng_te))
print(f'Stacking Classifier     →  {stacking_acc*100:.2f}%')

---
## Step 8 — Cross-Validation on Tuned Models

5-fold stratified CV on the **training set only**.  
Compare **CV mean ≈ Test accuracy** to detect overfitting:
- Large gap (CV >> Test) → overfit — model memorised training data
- Close scores → good generalisation

In [ ]:
print(f'{"Model":<22}  {"CV mean":>9}  {"CV std":>8}  {"Test acc":>10}  {"Overfit?":>9}')
print('-' * 65)

for name, r in tuned_results.items():
    scores = cross_val_score(r['model'], Xs_eng_tr, y_tr,
                             cv=CV5, scoring='accuracy', n_jobs=-1)
    gap    = abs(scores.mean() - r['test_acc']) * 100
    flag   = 'YES' if gap > 3 else 'no'
    print(f'{name:<22}  {scores.mean()*100:>8.2f}%  {scores.std()*100:>7.2f}%  '
          f'{r["test_acc"]*100:>9.2f}%  {flag:>9}  (gap={gap:.1f}pp)')

print('-' * 65)
print('\nNote: CV is computed on the training split (80%) only.')
print('      Test accuracy is the single 20% held-out evaluation.')

---
## Step 9 — Final Model Comparison

All models ranked by test accuracy — from baselines to tuned models to ensembles.

In [ ]:
all_scores = {}

# Baselines (best of 10 vs 13 features)
for name, r in baseline_results.items():
    fs_tag = '13f' if r['a13'] >= r['a10'] else '10f'
    all_scores[f'{name} (baseline, {fs_tag})'] = max(r['a10'], r['a13'])

# Tuned (all on 13 features)
for name, r in tuned_results.items():
    all_scores[f'{name} (tuned, 13f)'] = r['test_acc']

# Ensembles
all_scores['Voting   (ensemble, 13f)']  = voting_acc
all_scores['Stacking (ensemble, 13f)']  = stacking_acc

sorted_scores  = sorted(all_scores.items(), key=lambda x: x[1], reverse=True)
winner_name, winner_acc = sorted_scores[0]

print(f'{"Rank":<5}  {"Model":<40}  {"Test Accuracy":>14}')
print('=' * 64)
for i, (name, acc) in enumerate(sorted_scores, 1):
    tag = '  <-- WINNER' if i == 1 else ''
    print(f'{i:<5}  {name:<40}  {acc*100:>12.2f}%{tag}')
print('=' * 64)

print(f'\n--- Summary ---')
print(f'Majority-class baseline        : ~44.95%')
print(f'Previous best (RF, 10 features): 54.50%')
print(f'New winner    : {winner_name}')
print(f'New accuracy  : {winner_acc*100:.2f}%')
print(f'Improvement   : {(winner_acc - 0.545)*100:+.2f} percentage points')

## Detailed Report for Winner Model
Classification report (precision, recall, F1 per class) and confusion matrix.

In [ ]:
# Resolve winner model object
if 'Voting' in winner_name:
    winner_model = voting
elif 'Stacking' in winner_name:
    winner_model = stacking
elif '(tuned' in winner_name:
    raw_name = winner_name.split(' (tuned')[0]
    winner_model = tuned_results[raw_name]['model']
else:
    raw_name = winner_name.split(' (baseline')[0]
    use_eng  = baseline_results[raw_name]['a13'] >= baseline_results[raw_name]['a10']
    winner_model = baseline_results[raw_name]['m13'] if use_eng else baseline_results[raw_name]['m10']

y_pred = winner_model.predict(Xs_eng_te)

print(f'=== Classification Report: {winner_name} ===')
print(classification_report(y_te, y_pred, target_names=['Low', 'Medium', 'High']))

cm = confusion_matrix(y_te, y_pred)
print('Confusion Matrix (rows=actual, cols=predicted):')
print(f'          Low   Med  High')
for i, label in enumerate(['Low', 'Medium', 'High']):
    row = cm[i] if i < len(cm) else [0, 0, 0]
    print(f'  {label:7}  {row[0]:4}  {row[1]:4}  {row[2]:4}')

---
## Step 10 — Save Best Model

Persist three files to `models/`:
- `risk_model.pkl` — trained classifier
- `risk_scaler.pkl` — fitted StandardScaler (on 13 engineered features)
- `risk_meta.json` — metadata: algorithm name, accuracy, feature list

`predict.py` reads `risk_meta.json` at startup and automatically computes the 3 engineered  
features if `n_features >= 13`. No changes to the frontend or backend are needed.

In [ ]:
# Resolve final model and scaler
if 'Voting' in winner_name:
    final_model, final_scaler, final_feats = voting, scaler_eng, FEATURES_ENG
elif 'Stacking' in winner_name:
    final_model, final_scaler, final_feats = stacking, scaler_eng, FEATURES_ENG
elif '(tuned' in winner_name:
    raw = winner_name.split(' (tuned')[0]
    final_model, final_scaler, final_feats = tuned_results[raw]['model'], scaler_eng, FEATURES_ENG
else:
    raw     = winner_name.split(' (baseline')[0]
    use_eng = baseline_results[raw]['a13'] >= baseline_results[raw]['a10']
    final_model  = baseline_results[raw]['m13'] if use_eng else baseline_results[raw]['m10']
    final_scaler = scaler_eng if use_eng else scaler_base
    final_feats  = FEATURES_ENG if use_eng else FEATURES_BASE

# Save
joblib.dump(final_model,  MODELS_DIR / 'risk_model.pkl')
joblib.dump(final_scaler, MODELS_DIR / 'risk_scaler.pkl')

meta = {
    'algorithm'          : winner_name,
    'test_accuracy_pct'  : round(winner_acc * 100, 2),
    'features'           : final_feats,
    'n_features'         : len(final_feats),
    'engineered_features': ['budget_per_person', 'timeline_pressure', 'team_effectiveness'],
    'training_rows'      : int(len(y_tr)),
    'test_rows'          : int(len(y_te)),
}
with open(MODELS_DIR / 'risk_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved model  : {MODELS_DIR}/risk_model.pkl')
print(f'Saved scaler : {MODELS_DIR}/risk_scaler.pkl')
print(f'Saved meta   : {MODELS_DIR}/risk_meta.json')
print()
print(f'Algorithm : {winner_name}')
print(f'Accuracy  : {winner_acc*100:.2f}%')
print(f'Features  : {len(final_feats)}  →  {final_feats}')

---
## Summary

### ML Workflow Followed

```
Raw CSV (4,000 rows, 51 cols)
    │
    ▼
Select 10 feature columns + encode categoricals + merge Critical→High
    │
    ▼
Add 3 engineered features → 13 total
  • budget_per_person  = budget / team_size
  • timeline_pressure  = complexity / duration
  • team_effectiveness = experience × success_rate
    │
    ▼
Stratified 80/20 split  →  Train: 3,200 | Test: 800
    │
    ▼
StandardScaler (fit on train ONLY — no data leakage)
    │
    ├── 9 baseline models × 2 feature sets → 18 comparisons
    │
    ├── RandomizedSearchCV (n_iter=25, cv=5) on 5 top models
    │
    ├── Soft Voting + Stacking ensembles from top 3 tuned
    │
    └── 5-fold CV for overfitting check
    │
    ▼
Pick winner → save model + scaler + metadata
```

### How predict.py uses the saved model

1. Reads `risk_meta.json` → checks `n_features`
2. If `n_features == 13`: computes `budget_per_person`, `timeline_pressure`, `team_effectiveness` from the 10 user inputs
3. Applies saved scaler → calls `model.predict_proba()` → returns `{risk_level, score, probabilities}`

**No frontend or backend changes required** — the 3 engineered features are derived internally.

### To retrain the production model

```bash
cd ml
python train_risk_model.py
```

Or re-run this notebook to explore interactively and save the best result.